# Explore sensor CSV data from Google Drive

Load a Google Drive CSV into pandas, then inspect it with the same summaries and
plots as `01_explore_clean_data.ipynb`. Install `notebooks/requirements-drive.txt`
and select that environment's Python kernel. No SQL Server connection is needed.

Set `DRIVE_FILE_URL_OR_ID` in `.env` to a CSV shared with **Anyone with the
link**, with downloads allowed. No Google sign-in or access-mode setting is needed.
The notebook only reads the Drive file. Rerun the download cell to fetch changes.
It downloads the whole file before applying the date filter.

Use an uploaded **CSV**, not a Google Sheets document. Expect one timestamp
column and numeric sensor columns. Blank sensor readings stay missing; no
averaging or imputation is applied. Clear outputs before committing.


In [ ]:
from io import BytesIO
from pathlib import Path
import os
import re
from tempfile import TemporaryDirectory
from urllib.parse import parse_qs, urlencode, urlparse

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from dotenv import load_dotenv

sns.set_theme(style="whitegrid")

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "notebooks" / "01_explore_clean_data.ipynb").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch Jupyter from the repository root or notebooks directory.")

# Reload .env on reruns so edited Drive settings take effect.
load_dotenv(PROJECT_ROOT / ".env", override=True)


## Select the file and data window

Set `DRIVE_FILE_URL_OR_ID` in the repository `.env`.
The import cell reloads that file; rerun the import and configuration cells
after editing it. Set the timestamp header, CSV encoding, separator, and date
format below as needed. Match date boundaries to the CSV timezone.
The file needs download permission for **Anyone with the link**.


In [ ]:
DRIVE_FILE_URL_OR_ID = os.getenv("DRIVE_FILE_URL_OR_ID", "").strip()
TIMESTAMP_COLUMN = "datetime"  # Case-insensitive: also matches DateTime.
CSV_ENCODING = "utf-8-sig"  # Handles UTF-8 CSVs with or without a BOM.
CSV_SEPARATOR = ","
DATETIME_FORMAT = "mixed"   # Or an explicit format, e.g. "%m/%d/%Y %H:%M".
DAYFIRST = False            # True for day/month/year timestamps.

START_TIME = None  # Example: "2026-03-24 00:00:00" (inclusive).
END_TIME = None    # Example: "2026-04-24 00:00:00" (exclusive).

if not DRIVE_FILE_URL_OR_ID:
    raise ValueError("Set DRIVE_FILE_URL_OR_ID in the repository .env to your CSV link or ID.")
start = pd.Timestamp(START_TIME) if START_TIME is not None else None
end = pd.Timestamp(END_TIME) if END_TIME is not None else None
if (start is not None and pd.isna(start)) or (end is not None and pd.isna(end)):
    raise ValueError("Date boundaries must be valid timestamps or None.")
if start is not None and end is not None and start >= end:
    raise ValueError("START_TIME must be earlier than END_TIME.")


In [ ]:
def drive_file_id(value):
    """Accept an ID, /file/d/ID share link, or Drive URL with ?id=ID."""
    value = value.strip()
    if not value:
        raise ValueError("Set DRIVE_FILE_URL_OR_ID in .env to your CSV's Drive link or ID.")
    if re.fullmatch(r"[A-Za-z0-9_-]+", value):
        return value
    parsed = urlparse(value)
    if parsed.scheme != "https" or parsed.hostname != "drive.google.com":
        raise ValueError("Use a Google Drive file share link or file ID.")
    match = re.search(r"/file/d/([A-Za-z0-9_-]+)(?:/|$)", parsed.path)
    file_id = match.group(1) if match else parse_qs(parsed.query).get("id", [""])[0]
    if not re.fullmatch(r"[A-Za-z0-9_-]+", file_id):
        raise ValueError("The link must identify a file, not a folder or Google Sheet.")
    return file_id


def download_drive_csv(value):
    """Download a CSV shared with Anyone with the link, without signing in."""
    file_id = drive_file_id(value)
    import gdown

    # Use a temporary file so reruns never load a stale downloaded CSV.
    with TemporaryDirectory(prefix="drive-csv-") as directory:
        destination = str(Path(directory) / "sensor.csv")
        parameters = {"id": file_id}
        resource_key = parse_qs(urlparse(value.strip()).query).get("resourcekey", [None])[0]
        if resource_key:
            parameters["resourcekey"] = resource_key
        download_url = "https://drive.google.com/uc?" + urlencode(parameters)
        downloaded = gdown.download(
            url=download_url, output=destination, quiet=True, use_cookies=False,
        )
        if downloaded is None:
            raise RuntimeError(
                "Download failed. Check the link, sharing permissions, and download quota."
            )
        content = Path(downloaded).read_bytes()

    if content.lstrip().lower().startswith((b"<!doctype html", b"<html")):
        raise ValueError("Drive returned an HTML page rather than a CSV. Check file access.")
    return BytesIO(content)


In [ ]:
csv_stream = download_drive_csv(DRIVE_FILE_URL_OR_ID)
df = pd.read_csv(csv_stream, encoding=CSV_ENCODING, sep=CSV_SEPARATOR)
df.columns = df.columns.str.strip()
if not df.columns.is_unique:
    raise ValueError("CSV headers must be unique after trimming whitespace.")
# Match the timestamp header without changing sensor column names.
timestamp_matches = [
    column for column in df.columns
    if column.casefold() == TIMESTAMP_COLUMN.strip().casefold()
]
if not timestamp_matches:
    raise ValueError(
        f"Timestamp column {TIMESTAMP_COLUMN!r} is missing. Available columns: {list(df.columns)}"
    )
if len(timestamp_matches) > 1:
    raise ValueError(f"Multiple headers match the timestamp column: {timestamp_matches}")
timestamp_column = timestamp_matches[0]
if len(df.columns) < 2:
    raise ValueError("The CSV needs at least one sensor column besides the timestamp.")

df[timestamp_column] = pd.to_datetime(
    df[timestamp_column], format=DATETIME_FORMAT, dayfirst=DAYFIRST, errors="raise",
)
if df[timestamp_column].isna().any():
    raise ValueError("The CSV contains missing timestamps. Fix them before plotting.")
df = df.set_index(timestamp_column).sort_index()
df.index.name = "datetime"
if start is not None:
    df = df.loc[df.index >= start]
if end is not None:
    df = df.loc[df.index < end]

# Reject nonnumeric readings instead of silently changing them into missing values.
df = df.apply(pd.to_numeric, errors="raise")
print(f"Loaded {len(df):,} rows and {len(df.columns)} sensor columns.")
df.head()


## Inspect the dataset

Review the timestamp range, column types, missing readings, and summary statistics before choosing features or a training split. An empty result can indicate that the selected date window contains no imported data.


In [ ]:
df.info()
print("First timestamp:", df.index.min())
print("Last timestamp:", df.index.max())
print("Duplicate timestamps:", df.index.duplicated().sum())


In [ ]:
missing = pd.DataFrame({
    "missing_rows": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100),
})
missing.sort_values("missing_rows", ascending=False)


In [ ]:
df.describe()


## Null values per sensor

The chart includes every sensor, including columns with zero missing values. Labels show the exact null count and percentage of loaded rows. These counts describe missing CSV readings in existing rows; timestamps absent from the file are not counted.


In [ ]:
if df.empty or len(df.columns) == 0:
    print("No sensor rows to summarize. Check the selected data window.")
else:
    null_summary = missing.sort_values("missing_rows", ascending=False)
    fig, ax = plt.subplots(figsize=(12, max(4, 0.35 * len(null_summary))))
    sns.barplot(
        x=null_summary["missing_rows"].to_numpy(),
        y=null_summary.index.tolist(),
        color="steelblue", ax=ax,
    )
    for position, row in enumerate(null_summary.itertuples()):
        ax.annotate(
            f"{row.missing_rows:,} ({row.missing_percent:.1f}%)",
            (row.missing_rows, position), xytext=(5, 0),
            textcoords="offset points", va="center",
        )
    ax.set_xlim(0, max(1, null_summary["missing_rows"].max()) * 1.3)
    ax.set(title=f"Null readings per sensor ({len(df):,} rows)",
           xlabel="Number of null values", ylabel="Sensor")
    fig.tight_layout()
    plt.show()


## Sensor readings over time

Each sensor gets its own plot and value scale, with a shared time axis. By default, all sensor columns are plotted. Set `PLOT_COLUMNS` to a list such as `["A1_Qin", "A1_COD_in"]` to focus on selected sensors.

These are the original readings without averaging or imputation. Null readings break the line; missing timestamps can still be connected across a gap. Markers make individual observations visible, including isolated readings. An entirely null sensor has an empty plot.


In [ ]:
PLOT_COLUMNS = None  # None plots all sensors; or provide a list of column names.

plot_columns = list(df.columns) if PLOT_COLUMNS is None else list(PLOT_COLUMNS)
unknown_columns = [column for column in plot_columns if column not in df.columns]
if unknown_columns:
    raise ValueError(f"Unknown sensor columns: {unknown_columns}")

if df.empty or not plot_columns:
    print("No readings or sensor columns to plot. Check the data window and PLOT_COLUMNS.")
else:
    # Convert all-null object columns to numeric while retaining their missing values.
    plot_data = df[plot_columns].apply(pd.to_numeric, errors="raise")
    fig, axes = plt.subplots(
        len(plot_columns), 1, sharex=True,
        figsize=(14, max(3, 2.5 * len(plot_columns))), squeeze=False,
    )
    for ax, column in zip(axes[:, 0], plot_columns):
        ax.plot(plot_data.index, plot_data[column], linewidth=1,
                marker=".", markersize=2, color="steelblue")
        ax.set_title(column, loc="left")
        ax.set_ylabel("Value")
        if plot_data[column].isna().all():
            ax.text(0.5, 0.5, "All readings are null", transform=ax.transAxes,
                    ha="center", va="center")
    locator = mdates.AutoDateLocator()
    axes[-1, 0].xaxis.set_major_locator(locator)
    axes[-1, 0].xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    axes[-1, 0].set_xlabel("Timestamp")
    fig.suptitle("Cleaned sensor readings over time")
    fig.tight_layout(rect=(0, 0, 1, 0.98))
    plt.show()
